# SCASIA comparison

Set the results directory and named run folders below, then **Run All**. Rerun the last cell to refresh while experiments continue.
The notebook reads the existing evaluation files and writes the report under `comparison/`.
Missing runs and unfinished records are excluded; completed failures remain visible.

Keep this notebook beside `compare.py`. Start Jupyter from this directory as described in `README.md`.

In [ ]:
# Copyright (c) 2023 - 2026 Chair for Design Automation, TUM
# Copyright (c) 2025 - 2026 Munich Quantum Software Company GmbH
# All rights reserved.
#
# SPDX-License-Identifier: MIT
#
# Licensed under the MIT License

import runpy
from html import escape
from pathlib import Path

# Jupyter supplies the display tools outside the training environment.
from IPython import display as ipython_display  # ty: ignore[unresolved-import]

comparison = runpy.run_path("compare.py")

RESULTS = Path.home() / "scasia-runs"
# On this Mac: RESULTS = Path.home() / "Downloads/scasia-runs"
OUTPUT = RESULTS / "comparison"
RUNS = {
    "qiskit": RESULTS / "qiskit",
    "tket": RESULTS / "tket",
    "original": RESULTS / "original",
    "GNN earlier": RESULTS / "paper",
    "GNN warm start v1": RESULTS / "gnn-warmstart-v1/paper",
    "GNN warm start v2": RESULTS / "gnn-warmstart-v2/paper",
}
# Use RUNS = None for the original four compiler folders.

## Report and paper figure

The first plot recreates the layout of the manuscript's `reward_comp_esp_grouped.pdf`: ESP by algorithm, with named variants shown separately.
Qiskit and TKET are the static rows; **MQT Predictor** is `original`. The named GNN rows all use compiler mode `paper`.
The defaults compare the earlier GNN and warm-start variants. Missing evaluations are excluded.
Training settings and action registries may differ between variants; warnings show these differences.
This is not a controlled warm-start ablation. Inputs, calibration, objective, evaluation settings and pass timeout must match.

Only repetitions valid in every included row enter quality comparisons.
Each circuit contributes its mean over those repetitions; algorithm points average the circuit means.
The overall means give each circuit equal weight, and the plot orders algorithms by the paper row (or the mean of available rows).
There is no best-of-N selection. The figure uses your rerun results, not the manuscript's historical numbers.

The report also includes completion/failure counts, runtime, and paper-minus-original ESP where both rows are available.
SVG/PNG figures and summary/per-circuit CSV files are saved beside the HTML report.

In [ ]:
report = comparison["compare"](RESULTS, run_paths=RUNS)
if report["summary"]:
    comparison["write_report"](report, OUTPUT)
    page = escape((OUTPUT / "comparison.html").read_text(), quote=True)
    ipython_display.display(
        ipython_display.IFrame(
            src="about:blank",
            width="100%",
            height=1100,
            extras=[f'srcdoc="{page}"', 'title="SCASIA comparison"'],
        )
    )
    print(OUTPUT / "comparison.html")
else:
    print("\n".join(report["warnings"]))
    print("No completed evaluation records yet. Rerun this cell once results are available.")